# `main.ipynb` - Orquestador del pipeline Climate Engine

Este notebook **no contiene logica propia**: encadena los demas modulos,
en orden, con la magia `%run`, exactamente igual que en GeoStat y GeoKW.
Ejecutar este notebook de arriba a abajo ("Restart + Run All") corre el
pipeline completo: extraccion -> Bronze -> Silver -> Gold, para todas las
ciudades del catalogo.

> **Nota**: para ejecuciones selectivas por fase, ciudad o forzando el
> recalculo de particiones (requisito del enunciado de "seleccion
> selectiva de fases de calculo, localizaciones o forzado de
> particiones"), se usa el orquestador de linea de comandos `cli.py`, en
> la raiz del entregable (ver README). `cli.py` es deliberadamente un
> fichero minimo que importa estos mismos notebooks (via `importnb`) y
> expone sus fases como flags de CLI; la logica real vive aqui y en los
> modulos que este notebook encadena, nunca duplicada en `cli.py`.

> Depende de: todos los modulos de `climate_engine/`.

In [1]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")

## Encadenado de modulos

In [2]:
%run climate_engine/config.ipynb
%run climate_engine/logging_config.ipynb
%run climate_engine/utils_catalogo.ipynb
%run climate_engine/extract_openmeteo.ipynb
%run climate_engine/bronze_ingest.ipynb
%run climate_engine/silver_clean.ipynb
%run climate_engine/gold_analytics.ipynb

Fichero de configuracion: c:\Users\EM2026008667\Desktop\Proyectos internos\Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco\config.yaml
Raiz del proyecto:        c:\Users\EM2026008667\Desktop\Proyectos internos\Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Ventana de prediccion:    14 dias, zona horaria UTC
Politica de reintentos:   3 reintentos, backoff x2, codigos [429, 500, 502, 503, 504]
Rango temperatura valido: (-50.0, 60.0)
Rango humedad valido:     (0.0, 100.0)
Ruta bronze:  c:\Users\EM2026008667\Desktop\Proyectos internos\Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco\data\bronze  (existe: True)
Ruta silver:  c:\Users\EM2026008667\Desktop\Proyectos internos\Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco\data\silver  (existe: True)
Ruta

C:\Users\EM2026008667\AppData\Local\Temp\ipykernel_19052\1003325290.py:80: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: "; ".join(f"{r.column}: {r.check}" for r in g.itertuples()))


Filas crudas de entrada (incluye 1 corrupta inyectada a proposito): 1681
Filas validas:     1680
Filas en cuarentena: 1

Detalle de la cuarentena:
ciudad fecha_hora_utc  temperature_2m                        motivo_rechazo
Madrid     2026-09-23           999.0 temperature_2m: in_range(-50.0, 60.0)

Validacion del contrato de calidad de datos: OK

--- Ejecucion real completa de la capa Silver (sin el registro inyectado) ---
{'total_leido': 1680, 'validos': 1680, 'cuarentena': 0, 'filas_finales_en_silver': 1680}
Fichero de configuracion: c:\Users\EM2026008667\Desktop\Proyectos internos\Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco\config.yaml
Raiz del proyecto:        c:\Users\EM2026008667\Desktop\Proyectos internos\Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
V

## Ejecucion completa del pipeline

In [3]:
separador_inicio("INICIO DE EJECUCION DEL PIPELINE ETL - PROYECTO CLIMATE ENGINE")

validar_catalogo_minimo()
resumen_bronze = ejecutar_ingesta_bronze()
resumen_silver = procesar_silver(resumen_bronze)
resumen_gold = procesar_gold()

logger.info("EJECUCION FINALIZADA CORRECTAMENTE.")

print("=== RESUMEN DE LA EJECUCION ===")
print(f"id_lote:            {resumen_bronze['id_lote']}")
print(f"Origen por ciudad:  {resumen_bronze['origenes']}")
print(f"Silver - leidos:    {resumen_silver.get('total_leido', 0)}")
print(f"Silver - validos:   {resumen_silver.get('validos', 0)}")
print(f"Silver - cuarentena:{resumen_silver.get('cuarentena', 0)}")
print(f"Gold - tablas:      {resumen_gold}")

=== RESUMEN DE LA EJECUCION ===
id_lote:            104cfc2a4d86
Origen por ciudad:  {'Madrid': 'API', 'Barcelona': 'API', 'Valencia': 'API', 'Sevilla': 'API', 'Bilbao': 'API'}
Silver - leidos:    1680
Silver - validos:   1680
Silver - cuarentena:0
Gold - tablas:      {'resumen_diario': 70, 'media_movil_temp': 1680, 'zscore_horario': 1680}


---
✅ **`main.ipynb` (pipeline completo) verificado/a.**